# $ \text{Т.10} $

In [3]:
from scipy.special import kolmogorov
import numpy as np
import math

In [4]:
def F_norm(x, mean, sigma):
  """Функция нормального распределения."""
  return 0.5 * (1 + math.erf((x - mean) / (np.sqrt(2) * sigma)))

N = 100

m = np.array([5, 8, 6, 12, 14, 18, 11, 6, 13, 7])

## a)

In [5]:
F_emp = np.array([sum(m[:i]) for i in range(len(m) + 1)]) / N
F_th = np.arange(10) / 10

delta = np.sqrt(N) * np.max([max(np.abs(F_th[i] - F_emp[i]),
                                 np.abs(F_th[i] - F_emp[i + 1])) for i in range(F_th.size)])

p_value = kolmogorov(delta)

In [6]:
print("-" * 40)
print(f"p-value по критерию Колмогорова: {p_value}")
print("-" * 40)

----------------------------------------
p-value по критерию Колмогорова: 0.039681879538114355
----------------------------------------


## b)

In [7]:
segments = np.array([(-np.inf, 1)] + [(i, i + 1) for i in range(1, 9)] + [(9, np.inf)])

sample = np.repeat(np.arange(len(m)), m)

alpha = np.mean(sample)

sigma = np.sqrt(np.var(sample) * N / (N - 1))


def F_norm_wave(x): return F_norm(x, alpha, sigma)

P = [F_norm_wave(i[1]) - F_norm_wave(i[0]) for i in segments]

delta = sum((N * P[i] - m[i]) ** 2 / (N * P[i]) for i in range(10))

In [8]:
header = "i      | " + " | ".join(f"{i:<6}" for i in range(10))
print(header)
print("-" * len(header))

row_m = "m_i    | " + " | ".join(f"{val:<6}" for val in m)
print(row_m)

intervals = []
for i, seg in enumerate(segments):
    if i == 0 or i == len(segments) - 1:
        intervals.append(f"({seg[0]:.0f};{seg[1]:.0f})")
    else:
        intervals.append(f"[{seg[0]:.0f};{seg[1]:.0f})")
row_pgs = "П.Г.С. | " + " | ".join(f"{s:<6}" for s in intervals)
print(row_pgs)

# Печать nP_i
row_np = "nP_i   | " + " | ".join(f"{(p * N):<6.2f}" for p in P)
print(row_np)

print("-" * len(header))
print(f"Δ по О.М.П.Г. = {delta}")

i      | 0      | 1      | 2      | 3      | 4      | 5      | 6      | 7      | 8      | 9     
------------------------------------------------------------------------------------------------
m_i    | 5      | 8      | 6      | 12     | 14     | 18     | 11     | 6      | 13     | 7     
П.Г.С. | (-inf;1) | [1;2)  | [2;3)  | [3;4)  | [4;5)  | [5;6)  | [6;7)  | [7;8)  | [8;9)  | (9;inf)
nP_i   | 6.72   | 6.85   | 10.54  | 13.88  | 15.65  | 15.10  | 12.47  | 8.81   | 5.33   | 4.65  
------------------------------------------------------------------------------------------------
Δ по О.М.П.Г. = 16.871067048068763


In [9]:
bootstrap_iterations = 50_000

In [13]:
bootstrap_delta = []

x = np.arange(10)
delta_wave = np.sqrt(N) * np.max([max(np.abs(F_norm_wave(x[i]) - F_emp[i]),
                                      np.abs(F_norm_wave(x[i]) - F_emp[i + 1])) for i in x])

for _ in range(bootstrap_iterations):
  random_sample = np.array(sorted(np.random.normal(alpha, sigma, N)))

  alpha_bootstrap = random_sample.mean()

  sigma_bootstrap = np.sqrt(random_sample.var() * N / (N - 1))

  F_bootstrap_emp = [i / N for i in range(N + 1)]

  def F_bootstrap_wave(j):
    return F_norm(random_sample[j], alpha_bootstrap, sigma_bootstrap)

  sup = np.max([max(np.abs(F_bootstrap_wave(j) - F_bootstrap_emp[j]),
                    np.abs(F_bootstrap_wave(j) - F_bootstrap_emp[j + 1]))
                for j in range(len(random_sample))])

  bootstrap_delta.append(np.sqrt(N) * sup)

bootstrap_delta = np.array(bootstrap_delta)

p_value = len(bootstrap_delta[bootstrap_delta >= delta_wave]) / bootstrap_iterations

In [14]:
print("=" * 40)
print(f"p-value по критерию Колмогорова (Bootstrap): {p_value}")
print("=" * 40)

p-value по критерию Колмогорова (Bootstrap): 0.01474
